# Connect to St. Louis Federal Reserve Economic Data (FRED) API and retrieve data
This notebook will be used as an introductory notebook to start organizing the data for the St. Louis Federal reserve. This API will be used predominantly for the macroeconomic data that we will be pulling. It will likely have different time points, so I think I will need to pull the series indvidually to determine which can be concatenated together, we will then have to decide how we deal with heterogenous time series data.

*Note* - A free api key is required to access the data from the St. Louis Federal Reserve. If you need to generate an API key visit: https://fred.stlouisfed.org/docs/api/api_key.html

## Libraries

In [1]:
import numpy as np
import pandas as pd
import altair as alt

from fredapi import Fred

# Disable the max rows limit in Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

Now, that we have imported our libraries we need to connect to the fred_api. This requires our unique api_key and instantiating a fred instance using Fred. 

In [2]:
# Need an API key to access the FRED data
api_key = input("Enter your FRED API key: ")
fred = Fred(api_key=api_key)
print("Successfully connected to FRED API!")

Successfully connected to FRED API!


Alright, now let's start to pull data from the api. The hardest part of this is figuring out what the series name is that we want to pull. Let's start with something pretty straight forward like the Nominal GDP and Real GDP. This will give us a chance to see how far back we can obtain data as well. The nomial GDP should just be 'GDP' this is the unadjusted dollar value of Gross Domestic Product the real GDP should be under 'GDPC1', this is the inflation adjusted GDP in absolute US dollars.

In [3]:
# Call the fred api for GDP data
real_gdp = fred.get_series("GDPC1").dropna()
nom_gdp = fred.get_series("GDP").dropna()
print(f"We have {len(nom_gdp)} nominal GDP data points and {len(real_gdp)} real GDP data points.")

We have 316 nominal GDP data points and 316 real GDP data points.


Alright, we have 316 data points, considering these values are usually published quarterly this should be a significant amount of time so let's take a look at how far back this goes. 

In [4]:
nom_gdp.head()

1947-01-01    243.164
1947-04-01    245.968
1947-07-01    249.585
1947-10-01    259.745
1948-01-01    265.742
dtype: float64

Dating all the way back to 1947. Excellent. It does appear to be quarterly. Let's see if the most recent data lines up with what is published to verify the data and see whether or not the date is for the date the data was 'published' or more likely the start or end of the period the GDP is related to.

In [5]:
nom_gdp.tail()

2024-10-01    29825.182
2025-01-01    30042.113
2025-04-01    30485.729
2025-07-01    31098.027
2025-10-01    31490.070
dtype: float64

Okay, so the value corresponds to the BEGINNING of the period that the GDP is reported for. This is going to be a really important distinction to avoid data leakage. We wont have the values at the beginning of the quarter, likely we wont even have them until a quarter after. I'm pretty sure the Q4 2025 GDP numbers were just eleased today (2/2/26). Another thing that we need to think about is how these values are typically used. it's very rare to have absolute values used, so we may want to calculate both Quarter over Quarter (QoQ) changes and Year over Year changes (YoY).

In [ ]:
# First we need to convert to a dataframe
nom_gdp_df = nom_gdp.reset_index()
nom_gdp_df.columns = ["date", "nominal_gdp"]
# Calculate the quarter-over-quarter and year-over-year percentage changes
nom_gdp_df['QoQ_%'] = round(nom_gdp_df['nominal_gdp'].pct_change() * 100,2)
nom_gdp_df['YoY_%'] = round(nom_gdp_df['nominal_gdp'].pct_change(periods=4) * 100,2)
# Drop the NA values we created
nom_gdp_df.dropna(inplace=True)
# Let's reset the index  because we may merge on this later
nom_gdp_df.set_index("date", inplace=True)
# Let's take a look at the data to see if its accurate
nom_gdp_df.tail()

,nominal_gdp,QoQ_%,YoY_%
date,,,
2024-10-01,29825.182,1.06,4.93
2025-01-01,30042.113,0.73,4.65
2025-04-01,30485.729,1.48,4.59
2025-07-01,31098.027,2.01,5.38
2025-10-01,31490.070,1.26,5.58


Alright, this has been a great proof of concept ut now let's think about all of the different data that we may or may not want to use and write down their series name for ease.

### GDP and Debt
Nominal GDP = fred.get_series("GDP")  
Real GDP = fred.get_series("GDPC1")  
US Debt to GDP = fred.get_series("GFDEGDQ1885")  
Interest Payment on debt = fred.get_series("A091RC1Q027SBEA")  

### Inflation Data  
Consumer Price Index (CPI): fred.get_series("CPIAUCSL")  
Core PCI: fred.get_series("CPILFESL")  
Personal Consumption Expenditure (PCE): fred.get_series("PCEPI")  
Core PCE: fred.get_series("PCEPILFE")  
Producer Price Index (PPI): fred.get_series("PPIFIS")  

### Employment Data  
Unemployment Rate: fred.get_series("UNRATE")  
Initial Jobless Claims: fred.get_series("ICSA")  
Continued Jobless claims: fred.get_series("CCSA")  
Teenager unemployment rate: fred.get_series("LNS14000012")  
Adult unemployment rate: fred.get_series("LNS14000025")  
Male Unemployment rate: fred.get_series("LNS14000001")  
Female Unemployment rate: fred.get_series("LNS14000002")  
Average Duration of Unemployment: fred.get_series("UEMPMEAN")  

### Treasury Yield Data (cost of lending)  
1-Month Yield: fred.get_series("DGS1MO")  
3-Month Yield: fred.get_series("DGS3MO")  
6-Month Yield: fred.get_series("DGS6MO")  
1-Year Yield: fred.get_series("DGS1")  
2-Year Yield: fred.get_series("DGS2")  
5-Year Yield: fred.get_series("DGS5")  
10-Year Yield: fred.get_series("DGS10")  
30-Year Yield: fred.get_series("DGS30")  

These are the main one's I can think of for now but we can add to it as we see fit.